# Jupyter Notebook loading and creating an index of Third Place counts per PC4 
## Index documentation inside the code

## ------------------------------------
## Block 0 - packages
## ------------------------------------

In [ ]:
# Import required packages
import os
from pathlib import Path
os.environ['USE_PYGEOS'] = '0'

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Engine for all geopandas reads (fiona is broken on the secure machine).
GEO_ENGINE = "pyogrio"

## ------------------------------------
## Block 1 - Paths
## ------------------------------------

In [ ]:
# Change the path to the one where your scripts and data is saved

ROOT = Path("S:/")
export_path = ROOT / "the_tentative_team/results/"


## ------------------------------------
## Block 2 - Data loading validation
## ------------------------------------

In [ ]:
# Read the datafile that was linked in the previous step
linked_data = gpd.read_file(export_path / "linked_data.gpkg", engine=GEO_ENGINE)
linked_data.head()

# Data processing
## Create dataset with ranked third places

In [ ]:
ranked_data = linked_data
ranked_data.info()

In [ ]:
# Name the categories of third places for ranking them
TPtypes = [c for c in linked_data.columns if c.startswith("third_places_")]
TPtypes

In [ ]:
# Rank all PC4s by the number of third places. 
# When several PC4s have 0 third places they are given a higher rank together. 
# To make sure that the lowest ranking is 0, we subtract the minimum rank from it.
# Then, we divide the rank by the maximum and multiply by 100. The resulting score runs from 0 to 100.
for col in TPtypes:
    a = linked_data[[col]].rank(ascending = True)
    b = a-a.min()
    ranked_data[col + "_ranked"] = b/b.max()*100
ranked_data.head()

In [ ]:
# Similar to above, but now the ranking is done within 5 categories of urbanisation ("stedelijkheid")
for col in TPtypes:
    ranked_data["ranking"] = ranked_data.groupby("stedelijkheid")[[col]].rank(ascending = True)
    rankmin = ranked_data.groupby("stedelijkheid")["ranking"].min()
    ranked_data["ranking"] = np.where(ranked_data["stedelijkheid"] == 1, ranked_data["ranking"] - rankmin[1], 
                                                          np.where(ranked_data["stedelijkheid"] == 2, ranked_data["ranking"] - rankmin[2], 
                                                                  np.where(ranked_data["stedelijkheid"] == 3, ranked_data["ranking"] - rankmin[3], 
                                                                          np.where(ranked_data["stedelijkheid"] == 4, ranked_data["ranking"] - rankmin[4], 
                                                                                   ranked_data["ranking"] - rankmin[5]))))
    
    rankmax = ranked_data.groupby("stedelijkheid")["ranking"].max()
    ranked_data[col + "_ranked_by_urbanisation"] = np.where(ranked_data["stedelijkheid"] == 1, ranked_data["ranking"]/rankmax[1]*100, 
                                                          np.where(ranked_data["stedelijkheid"] == 2, ranked_data["ranking"]/rankmax[2]*100, 
                                                                  np.where(ranked_data["stedelijkheid"] == 3, ranked_data["ranking"]/rankmax[3]*100, 
                                                                          np.where(ranked_data["stedelijkheid"] == 4, ranked_data["ranking"]/rankmax[4]*100, 
                                                                                  ranked_data["ranking"]/rankmax[5]*100))))
ranked_data.head()

In [ ]:
# Drop the columns with the number of third places instead of the index:
ranked_data = ranked_data.drop(labels = TPtypes, axis = 1)
ranked_data = ranked_data.drop(labels = ["ranking"], axis = 1)
ranked_data.info()

## Data exporting

In [ ]:
ranked_data.to_file(export_path / "indexed_data.gpkg" , driver = "GPKG")

In [ ]:
ranked_data.to_csv(export_path /"indexed_data.csv", index = False)